## Multi-Class Prediction of Obesity Risk

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

# -----------------------------
# Load Data
# -----------------------------
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Separate target
y = train["NObeyesdad"]
X = train.drop(columns=["NObeyesdad", "id"])

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)

# One-hot encode predictors
X = pd.get_dummies(X)
test_ids = test["id"]
test = pd.get_dummies(test.drop(columns=["id"]))

# Align test columns
X, test = X.align(test, join="left", axis=1, fill_value=0)

# Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# -----------------------------
# 1. Decision Tree
# -----------------------------
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_val)
dt_acc = accuracy_score(y_val, dt_pred)

# -----------------------------
# 2. Bagging
# -----------------------------
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=300,
    random_state=42
)
bag.fit(X_train, y_train)
bag_pred = bag.predict(X_val)
bag_acc = accuracy_score(y_val, bag_pred)

# -----------------------------
# 3. Random Forest
# -----------------------------
rf = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    random_state=42
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_val)
rf_acc = accuracy_score(y_val, rf_pred)

# -----------------------------
# 4. Boosting
# -----------------------------
boost = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
boost.fit(X_train, y_train)
boost_pred = boost.predict(X_val)
boost_acc = accuracy_score(y_val, boost_pred)

print("Decision Tree:", dt_acc)
print("Bagging:", bag_acc)
print("Random Forest:", rf_acc)
print("Boosting:", boost_acc)

# -----------------------------
# Final Kaggle Submission
# -----------------------------
best_model = rf  # replace with best performing model
final_predictions = best_model.predict(test)

submission = pd.DataFrame({
    "id": test_ids,
    "NObeyesdad": le.inverse_transform(final_predictions)
})

submission.to_csv("Assignment_6_submission.csv", index=False)

Decision Tree: 0.8712267180475273
Bagging: 0.8919396274887604
Random Forest: 0.8876043673731535
Boosting: 0.9035003211303789
